## EDA — Factibilidad y objetivos

Se justifica si cada técnica es factible con los datos en `data/staging`, y qué se espera hallar con cada una

---
### 1. Factibilidad

#### a. Similitud (Jaccard, MinHash, LSH)

- `descripcion`: 0% nula, mediana 29 palabras, ~81% distintas

- `items_desc`: 100% poblado, mediana 24 palabras; se concatena con `descripcion` en `texto` antes de generar shingles

- 9,276 grupos (20,711 filas, ~8.8% del dataset) comparten entidad+fecha+monto+descripción

- **Columnas:** `descripcion`, `items_desc` → `texto` → `shingles`; `entidad`, `fecha`

- **Nota `k`:** `generar_shingles` usa `k=5` (basado en literatura)

- **Veredicto:** Factible — texto largo y variado, con evidencia concreta de casi-duplicados

#### b. Vecinos aproximados (ANN: IVF, HNSW)

- `descripcion`/`items_desc` con la misma cobertura que en (a), listas para TF-IDF vía `tokens`

- `monto_referencial` 0% nulo; `monto_final` lo hereda — casi todos los procesos tienen monto comparable

- El monto varía mucho por `categoria` (mediana S/150-168k en *goods*/*services* vs S/1.4M en *works*); 
conviene comparar vecinos dentro de la misma categoría

- **Columnas:** `descripcion`, `items_desc` → `tokens`; `monto_final`; `categoria`

- **Veredicto:** Factible, con la salvedad de segmentar por categoría antes de buscar vecinos

#### c. Minería de flujos (Bloom Filter, Count-Min Sketch, DGIM)

- 236,229 procesos con fecha continua entre 2023-2025, sin huecos grandes

- 66,141 proveedores distintos; el top-10 concentra solo ~5.2% del monto adjudicado en general

- A nivel de **par** entidad-proveedor sí hay reincidencia fuerte: PROMPERÚ-Meta 155 veces, PROMPERÚ-Google 128, PROMPERÚ-TikTok 113, pares ESSALUD-proveedor médico 73/62/59

- Caso de uso de Bloom Filter ("¿ya vi este par antes?") y Count-Min Sketch (frecuencia aproximada) frente a guardar cada par exacto en memoria

- **Columnas:** `fecha`; `entidad`, `proveedor`; `n_postores` → `postor_unico`

- **Veredicto:** Factible — flujo largo y continuo, con pares entidad-proveedor con cientos de
  repeticiones

#### d. Reglas de asociación (Apriori, FP-Growth)

- `categoria`, `metodo_detalle`, `region` y tramo de monto: cardinalidad suficiente para ítems de
  canasta

- `metodo_detalle`: Contratación Directa 86.5%, Régimen Especial 80.0%, Convenio 54.6% de postor único
  vs 0.4-2.1% en métodos competitivos

- Tramo de monto: 57.3% de postor único en `1_<50k` vs 6.9-9.8% en el resto

- `region`: Tumbes 30.7%, Lima 26.3% en postores unicos (sobre 68,658 procesos, no es muestra chica)

- **Columnas:** `categoria`, `metodo_detalle`, `region`, `tramo_monto`, `rubro`, `n_postores` →
  `postor_unico` (vía `canasta`)

- **Veredicto:** Factible — las señales `{Contratación Directa}`, `{tramo bajo}` y `{región Lima/Tumbes}`
  → postor único ya son visibles a simple vista; Apriori/FP-Growth debería formalizarlas

### 2. Objetivos

#### a. Similitud

- Procesos con descripción casi idéntica, de una misma entidad y en fechas cercanas — indicio de **fraccionamiento**

- **Pregunta de negocio:** P2

- **Columnas:** `shingles`, `entidad`, `fecha`.

#### b. Vecinos aproximados

- Para un proceso dado, los *k* procesos más parecidos por objeto de compra (texto)

- Comparar cuánto varía el monto contratado entre entidades/regiones — **precios de referencia**.

- **Pregunta de negocio:** P3 

- **Columnas:** `tokens`, `monto_final`.

#### c. Minería de flujos

- Cuántos procesos con un solo postor hay en la ventana reciente del flujo (DGIM)

- Si un par entidad-proveedor ya se repitió antes (Bloom/Count-Min) — señales de **baja competencia** y reincidencia 

- **Pregunta de negocio:** P1 

- **Columnas:** `postor_unico`, `fecha`, `entidad`, `proveedor`.

#### d. Reglas de asociación

- Combinaciones de método + tramo de monto + región que se asocian fuertemente a **postor único** (baja competencia)

- Reglas tipo `{método, tramo de monto} → {postor único}`

- **Pregunta de negocio:** P1 

- **Columnas:** `canasta`.

### 3. Resumen

| Técnica | Pregunta | Columnas clave |
|---|---|---|
| Similitud (Jaccard/MinHash/LSH) | P2 — Fraccionamiento | `shingles`, `entidad`, `fecha` |
| Vecinos aproximados (ANN) | P3 — Precios de referencia | `tokens`, `monto_final` |
| Minería de flujos (Bloom/Count-Min/DGIM) | P1 — Competencia | `postor_unico`, `fecha`, `entidad`, `proveedor` |
| Reglas de asociación (Apriori/FP-Growth) | P1 — Competencia | `canasta` |